# Tools and Agents (open source): Function Calling in LangChain

### Outline
- A quick Pydantic syntax primer (plain vs. validated classes)
- A Pydantic model as a tool schema, via `bind_tools`
- Forcing a tool call - and why that doesn't work on Ollama
- Using a bound model in a chain, and with multiple tools

Open-source recode of `03-Functions-Tools-and-Agents-with-LangChain/L3-function-calling-student.ipynb`.
The original notebook hand-converts Pydantic models to OpenAI function specs via
`convert_pydantic_to_openai_function`, then does `model.bind(functions=...)`. That helper and
`.bind(functions=...)` were both removed in modern LangChain: you now pass Pydantic models (or
plain callables, or `BaseTool` instances) straight into `model.bind_tools([...])`, and LangChain
converts them under the hood. The Pydantic-syntax primer at the top is unchanged - it's plain
Pydantic, nothing OpenAI- or Ollama-specific about it.

## Setup

This notebook is the open-source / Ollama-cloud recode of the matching lesson in
[`openai_agentic_ai_course`](../../openai_agentic_ai_course/), reusing the shared
`common.py` / `tracing.py` helpers already built for
[`tools_and_agent`](../../tools_and_agent/) rather than duplicating them here.

- **Model**: `ChatOllama`, pointed at the Ollama cloud endpoint configured in the
  repo-root `.env` (`OLLAMA_MODEL` / `OLLAMA_BASE_URL` / `OLLAMA_API_KEY`).
- **Tracing**: every `.invoke()` / `.batch()` / `.stream()` call below passes
  `config=traced("run name")`, which attaches a Langfuse callback - open the
  Langfuse dashboard and filter by trace name to see this notebook's calls.
- **Kernel**: run this with the repo's `.venv` (`Python 3 (ipykernel)`) - it already
  has everything in [`requirements.txt`](../../requirements.txt) installed.

In [ ]:
import sys
from pathlib import Path

# common.py / tracing.py live in tools_and_agent/, not here - add it to sys.path
# instead of copying them, so this notebook always uses the one shared implementation.
COURSE_DIR = Path("../../tools_and_agent").resolve()
if str(COURSE_DIR) not in sys.path:
    sys.path.insert(0, str(COURSE_DIR))

from typing import Optional

from common import get_model, traced
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field, ValidationError

## Pydantic syntax

A plain Python class does no validation at all - assigning a string to an `int`-typed attribute
just works.

In [ ]:
class User:
    def __init__(self, name: str, age: int, email: str):
        self.name = name
        self.age = age
        self.email = email


foo = User(name="Joe", age=32, email="joe@gmail.com")
print("plain class age (no validation):", User(name="Joe", age="bar", email="joe@gmail.com").age)

In [ ]:
class pUser(BaseModel):
    name: str
    age: int
    email: str


foo_p = pUser(name="Jane", age=32, email="jane@gmail.com")
print("pydantic name:", foo_p.name)

In [ ]:
# This raises - "bar" is not a valid int, and Pydantic actually enforces the annotation.
try:
    pUser(name="Jane", age="bar", email="jane@gmail.com")
except ValidationError as e:
    print(e)

In [ ]:
class Class(BaseModel):
    students: list[pUser]


obj = Class(students=[pUser(name="Jane", age=32, email="jane@gmail.com")])
obj

## Pydantic model as a tool schema

A Pydantic model's docstring becomes the tool's description, and its fields become the tool's
parameters - `bind_tools` converts it to a JSON schema under the hood.

In [ ]:
class WeatherSearch(BaseModel):
    """Call this with an airport code to get the weather at that airport"""

    airport_code: str = Field(description="airport code to get weather for")


model = get_model()

model_with_function = model.bind_tools([WeatherSearch])
result = model_with_function.invoke("what is the weather in sf?", config=traced("L3: single Pydantic tool"))
result.tool_calls

## Forcing a tool call

The original notebook forces OpenAI to call a specific function via `function_call={"name": ...}`.
`bind_tools(..., tool_choice=...)` accepts the same idea, but `langchain-ollama` documents it as
a no-op: "This parameter is currently ignored as it is not supported by Ollama." Proven below -
`hi!` gets no tool call either way. The practical workaround, same one used in L1, is to only
offer the one tool you want called.

In [ ]:
model_with_forced_function = model.bind_tools([WeatherSearch], tool_choice="WeatherSearch")
result = model_with_forced_function.invoke("hi!", config=traced("L3: forced tool_choice (ignored)"))
print("tool_calls (expect empty - tool_choice was not honored):", result.tool_calls)

## Using it in a chain

`prompt | model_with_function` - the bound model slots into LCEL like any other model.

In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant"),
        ("user", "{input}"),
    ]
)
chain = prompt | model_with_function

result = chain.invoke({"input": "what is the weather in sf?"}, config=traced("L3: prompt | model_with_function"))
result.tool_calls

## Using multiple tools

Bind two schemas and let the model pick the right one - or neither, for a plain greeting.

In [ ]:
class ArtistSearch(BaseModel):
    """Call this to get the names of songs by a particular artist"""

    artist_name: str = Field(description="name of artist to look up")
    n: int = Field(description="number of results")


model_with_functions = model.bind_tools([WeatherSearch, ArtistSearch])

print(model_with_functions.invoke("what is the weather in sf?", config=traced("L3: multi-tool weather")).tool_calls)

In [ ]:
print(
    model_with_functions.invoke(
        "what are three songs by taylor swift?", config=traced("L3: multi-tool artist")
    ).tool_calls
)

In [ ]:
result = model_with_functions.invoke("hi!", config=traced("L3: multi-tool greeting"))
print("content:", result.content)
print("tool_calls:", result.tool_calls)